In [ ]:
# --- CELDA 1: INSTALACIÓN Y DESCARGA ---
!pip install ultralytics roboflow -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from roboflow import Roboflow
rf = Roboflow(api_key="(busquen su API key en https://roboflow.com/account)")
project = rf.workspace("edwin-workspace").project("crosswalk-g7tpd")
version = project.version(2)
dataset = version.download("yolov8-obb")

import os
DATA_YAML = os.path.join(dataset.location, "data.yaml")

# Verifica que salgan las 2 clases
with open(DATA_YAML, "r") as f:
    print(f.read())

# --- CELDA 2: ENTRENAMIENTO ---
from ultralytics import YOLO

# Con A100 puedes usar yolov8l sin problema (más preciso)
# Si quieres mantener yolov8m para luego exportar a Jetson, déjalo
model = YOLO("yolov8m.pt")   # o "yolov8l.pt" si quieres más precisión

results = model.train(
    # ── Dataset ──
    data=DATA_YAML,

    # ── Épocas ──
    epochs=200,
    patience=20,

    # ── Imagen y batch (A100 aguanta mucho) ──
    imgsz=640,
    batch=64,              # ⬆ A100 puede con 64 en yolov8m fácil (prueba 128 si quieres)
    workers=8,             # ⬆ más hilos de carga

    # ── Optimizador ──
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,

    # ── Hardware ──
    device=0,
    amp=True,              # ✅ Mixed precision, A100 tiene Tensor Cores (más rápido)
    cache="ram",           # ✅ Cachea dataset en RAM (Colab A100 tiene ~80GB RAM)



    # ── Guardar ──
    project="Traxxas",
    name="crosswalk_stop_v1",
    save=True,
    save_period=10,

    plots=True,
    verbose=True,
)

print("\n✅ Entrenamiento terminado")

# --- CELDA 3: EVALUACIÓN ---
best_model = YOLO("Traxxas/crosswalk_stop_v1/weights/best.pt")

metrics = best_model.val(data=DATA_YAML, imgsz=640)

print(f"\n📈 MÉTRICAS GLOBALES")
print(f"  mAP50     : {metrics.box.map50:.4f}")
print(f"  mAP50-95  : {metrics.box.map:.4f}")

# Métricas por clase (MUY útil con 2 clases para ver si una va peor)
print(f"\n📈 POR CLASE")
names = best_model.names
for i, cls_name in names.items():
    print(f"  {cls_name:15s}  mAP50={metrics.box.maps[i]:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 101.5 MB/s eta 0:00:00
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Crosswalk-2 in yolov8-obb:: 100%|██████████| 79933/79933 [00:10<00:00, 7973.77it/s] 


train: train/images
val: valid/images
test: test/images

names: 
  0: Stop
  1: crosswalk
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Crosswalk-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=

FileNotFoundError: [Errno 2] No such file or directory: 'Traxxas/crosswalk_stop_v1/weights/best.pt'